# Lab 4

<table class="tfo-notebook-buttons" align="left">
  <td>
    <a target="_blank" href="https://colab.research.google.com/github/taipeitechmmslab/MMSLAB-Pytorch/blob/master/Lab4.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />Run in Google Colab</a>
  </td>
  <td>
    <a target="_blank" href="https://github.com/taipeitechmmslab/MMSLAB-Pytorch/blob/master/Lab4.ipynb"><img src="https://www.tensorflow.org/images/GitHub-Mark-32px.png" />View source on GitHub</a>
  </td>
</table>

### Import

In [ ]:
!pip install torchinfo

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
import torchvision
import torchvision.transforms as T
from torchinfo import summary
import matplotlib.pyplot as plt
from PIL import Image
import urllib.request
import os
import numpy as np
from tqdm import tqdm


### 讀取數據並分析
使用Torchvision datasets載入Cifar10數據集

In [ ]:
# 定義套用到資料集的預處理, 這邊我們僅將圖片轉換成 pytorch 的 tensor 形式
transform = T.ToTensor()
# 從 torchvision.datasets載入CIFAR10資料集
dataset = torchvision.datasets.CIFAR10(root='./data', train=True, transform=transform, download=True)


顯示Cifar10數據集資訊

In [ ]:
print(f'data shape in dataset: {dataset.data.shape}')
print(f'class in dataset: {dataset.classes}')


Cifar10的十個類別及其數量


In [ ]:
# 我們可以透過np.unique來進行類別數量的計算
# CIFAR-10 的標籤直接存放在 dataset.targets 中，我們直接將其轉換成 numpy 陣列
targets = np.array(dataset.targets)

# 透過 np.unique 計算每個類別出現的次數
classes, counts = np.unique(targets, return_counts=True)
print((classes, counts))

顯示數據集部分影像資料：

In [ ]:
# 為了方便抓取一批隨機影像來顯示，我們將完整的 dataset 放入基礎的 DataLoader 中
temp_loader = torch.utils.data.DataLoader(dataset, batch_size=128, shuffle=True)

# 先定義欲顯示影像的數量 (11 x 11)
grid_num = 11

# 建立 iterator 並透過 next 方法抓取一個 batch
dataiter = iter(temp_loader)
images, labels = next(dataiter)

# 利用 torchvision 的 make_grid 方法建立一個顯示影像的陣列
grid_example_img = torchvision.utils.make_grid(images[:grid_num ** 2], nrow=grid_num, value_range=(0, 1), normalize=True)

# 透過 matplotlib.pyplot 顯示影像
plt.figure(figsize=(10, 10))
# 加上 .numpy() 確保 Tensor 正確轉換為 Numpy 陣列供 matplotlib 繪圖
plt.imshow(np.transpose(grid_example_img.numpy(), (1, 2, 0)))
plt.show()


### Dataset 設定

資料愈處理(Data Prepossessing ):
- 影像資料：將輸入影像做標準化，即將影像中的所有數值減去0.5，再將資料的標準差設定為0.5。

In [ ]:
batch_size = 128
# 定義 dataloader 的 cpu 數量
cpu_num = 4 if os.cpu_count() > 4 else os.cpu_count()
if os.name == 'nt':
    # Windows 系統的 dataloader 使用大於 0 的 cpu_num 時可能會變很慢,
    # 因此這邊用 os.name 判斷作業系統, 若為 windows 則將 cpu 數量設為 0
    cpu_num = 0

# 定義未使用 image augmentation 的 dataset 相關參數
transform = T.Compose([
    T.ToTensor(),
    T.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])
dataset = torchvision.datasets.CIFAR10(root='./data', train=True, transform=transform, download=True)
d_len = len(dataset)
# 透過 random_split 將 dataset 分成 training set 和 validation set
trainset, validset = torch.utils.data.random_split(dataset, [int(d_len * 0.7), int(d_len * 0.3)])
trainloader = DataLoader(trainset, batch_size=batch_size, shuffle=True, num_workers= cpu_num)
validloader = DataLoader(validset, batch_size=batch_size, shuffle=False, num_workers= cpu_num, pin_memory=True)

# 定義 image augmentation 會用到的 transformations, 並將其套用至 dataset 中
transform = T.Compose([
    T.RandomAffine(degrees=(-10, 10), translate=(0.1, 0.2), scale=(0.9, 1)),
    T.RandomCrop(size=30),
    T.Resize(size=32),
    T.ColorJitter(brightness=(.8, 1.1), contrast=.1, saturation=.3),
    T.ToTensor(),
    T.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

# 創建使用 image augmentation 的 dataset 和 dataloader
dataset_aug = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
trainset_aug, _ = torch.utils.data.random_split(dataset_aug, [int(d_len * .7), int(d_len * .3)])
trainloader_aug = torch.utils.data.DataLoader(trainset_aug, batch_size=batch_size, shuffle= True, num_workers=cpu_num, pin_memory=True)


### 資料增強練習

下載圖片

In [ ]:
# 引入下載檔案用的套件
import urllib.request

# 定義圖片的下載網址與本機存檔路徑
image_url = "https://i.imgur.com/gjeDs00.png"
file_path = "./sample_image.png"

if not os.path.exists(file_path):
    # 實務上，許多圖片網站會阻擋程式碼的自動下載。因此我們透過加入 User-agent，模擬瀏覽器瀏覽，藉此順利取得影像資料。
    opener = urllib.request.build_opener()
    opener.addheaders = [('User-agent', 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)')]
    urllib.request.install_opener(opener)
    
    # 透過 os.path.exists 檢查圖片是否已經存在
    # 若不存在則透過 urllib.request.urlretrieve 來下載影像, 並存到指定路徑 (file_path) 中
    urllib.request.urlretrieve(image_url, file_path)

# 使用 PIL 的 Image 套件讀取影像資料
img = Image.open(file_path)

# 透過 matplotlib.pyplot 顯示影像
plt.figure()
plt.imshow(img)
plt.show()

將影像轉換成 tensor，並定義要套用的 augmentation functions和可視化設定

In [ ]:
# 將影像轉換成 tensor
img = T.ToTensor()(img)

# 定義 random transforms 和對應的名稱
# 分別有 隨機水平翻轉影像、隨機改變顏色、隨機旋轉圖片、隨機放大縮小圖片
transform_list = [
  ['Random Flip', T.RandomHorizontalFlip(p=.75)],
  ['Random Color', T.ColorJitter(brightness=(.75, 1.25), hue=.1, contrast=.3, saturation=.3)],
  ['Random Rotation', T.RandomRotation(degrees=(-45, 45))],
  ['Random Scale', T.RandomAffine(degrees=(0, 0), translate=(0, 0), scale=(0.5, .85))],
]

# 設定一個 random augmentation 要可視化幾次
n = 4


使用matplotlib可視化結果

In [ ]:
# 透過 matplotlib 套件創建一個 len(transform_list) x n 的圖片格, figize 是用來設定最終圖片的長寬
fig, ax = plt.subplots(len(transform_list), n + 1, figsize=(12.5, 13.5), clear=True)
# 由於 torch 和 torchvision 的影像使用 [C, H, W] 的排列, 而 matplotlib 是使用 [H, W, C]
# 因此需要一個修復通道順序的函式, 這邊使用 permute 來交換通道順序
# 輸入的 [C, H, W] 分別是為 0, 1, 2
# 輸出要變成 [H, W, C], 因此給的參數是 (1, 2, 0)
dim_fixup = lambda img_in: torch.permute(img_in, (1, 2, 0))

# 透過matplotlib畫圖，ax的維度是[4, n]，每個ax代表圖片網格的第[y,x]個圖片
for index, (trans_name, trans_func) in enumerate(transform_list):
  ax[index, 0].imshow(dim_fixup(img))
  ax[index, 0].set_title('Original')
  for x in range(1, n + 1):
    ax[index, x].imshow(dim_fixup(trans_func(img)))
    ax[index, x].set_title(trans_name)

plt.tight_layout()
plt.show()


### 建立全連接神經網路

建立網路模型，這邊使用到以下幾種網路層：
-	nn.faltten：扁平層（特徵圖轉成一維Tensor）。
-	nn.Dropout：Dropout層（每次訓練隨機丟棄30%神經元）。
-	nn.Linear：全連接層（隱藏層使用ReLU激活函數，輸出層使用Softmax激活函數）。


In [ ]:
class FullyConnectedModel(nn.Module):
    """" 全連接網路架構 """
    def __init__(self, dropout_prob=0.3):
        super().__init__()
        self.flatten = nn.Flatten()
        # 使用 nn.Linear 定義線性層，並將其加入 ModuleList 中
        self.dense = nn.ModuleList([
            nn.Linear(32 * 32 * 3, 128, bias=True),
            nn.Linear(128, 256, bias=True),
            nn.Linear(256, 512, bias=True),
            nn.Linear(512, 1024, bias=True),
            nn.Linear(1024, 512, bias=True),
            nn.Linear(512, 256, bias=True),
            nn.Linear(256, 64, bias=True),
            nn.Linear(64, 10, bias=False),  # 最後一層不使用 bias
        ])
        # 定義激活函數和 dropout
        self.relu = nn.ReLU()
        # 定義 dropout
        self.dropouts = nn.ModuleList([
            nn.Dropout(dropout_prob) for _ in range(len(self.dense) - 1)
        ])

    def forward(self, x):
        # 全連接網路會將圖片攤平成一維向量，因此需要使用 flatten
        x = self.flatten(x)
        # 使用 for 迴圈將線性層和激活函數進行連接
        for i in range(len(self.dense) - 1):
            x = self.dense[i](x)
            x = self.relu(x)
            x = self.dropouts[i](x)
        # 最後一層不使用激活函數
        x = self.dense[-1](x)
        return x


### 建立卷積神經網路

建立網路模型，這邊使用到以下幾種網路層：
-	nn.Conv2D：卷積層（使用ReLU激活函數，以及3x3大小的kernel）。
-	nn.MaxPool2D：池化層（對特徵圖下採樣）。
-	nn.Flatten：扁平層（特徵圖轉成一維Tensor）。
-	nn.Dropout：Dropout層（每次訓練隨機丟棄50%神經元）。
-	nn.Linear：全連接層（隱藏層使用ReLU激活函數，輸出層使用Softmax激活函數）。


In [ ]:
class ConvolutionalModel(nn.Module):
    """ 卷積神經網路 """
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        # 這次使用 nn.Lazy 系列的函數，主要差別僅在於不需要指定輸入的 channel 數量
        self.convs = nn.ModuleList([
            nn.LazyConv2d(64, (3, 3),  padding=1, bias=True),
            nn.LazyConv2d(128, (3, 3), padding=1, bias=True),
            nn.LazyConv2d(256, (3, 3), padding=1, bias=True),
            nn.LazyConv2d(256, (3, 3), padding=1, bias=True),
            nn.LazyConv2d(128, (3, 3), padding=1, bias=True),
            nn.LazyConv2d(64, (3, 3),  padding=1, bias=True),
        ])
        # 定義線性層, 此處也使用 nn.LazyLinear 來定義
        self.dense = nn.LazyLinear(64, bias=True)
        self.output_layer = nn.LazyLinear(10, bias=False)
        self.relu = nn.ReLU()
        self.pool = F.max_pool2d

    def forward(self, x):
        # 使用 for 迴圈將卷積層和激活函數進行連接
        for i in range(len(self.convs)):
            x = self.convs[i](x)
            x = self.relu(x)
            if i == 0:
                # 第一層卷積層的輸出使用 max pooling
                x = self.pool(x, 3, 2)
        # 將輸出攤平成一維向量, 並使用線性層進行轉換
        x = self.flatten(x)
        x = self.dense(x)
        x = self.output_layer(x)
        return x


### 訓練與測試模型

定義輔助訓練相關的函數

In [ ]:
def metric(x, y):
    """ 計算預測結果的 accuracy """
    # 使用 torch.argmax 將預測結果轉換成類別的 index
    x = torch.argmax(x, 1)
    # 使用 torch.sum 將預測正確的數量加總, 並除以總數量得到 accuracy
    accu = torch.sum(x == y) / y.shape[0] * 100
    return accu


def compute_loss_and_accuracy(batch, net, loss_func, device=None):
    """ 根據傳入的 loss function 和資料計算一個 batch 的 loss 和 accuracy """
    x, y = batch
    x = x.to(device)
    y = y.to(device)
    # 透過 forward pass 得到輸出
    outputs = net(x)
    return loss_func(outputs, y), metric(outputs, y)


設定訓練與測試的函數

In [ ]:

def evaluate(net, data_loader, loss_function, device):
    """ 對整個網路進行 validation set 的準確度判定 """
    # 將網路透過 .eval() 轉換成 evaluation 模式, 這會影響到 dropout 和 batch normalization 等層的行為
    net.eval()
    loss_total, accu_total = 0, 0
    
    # 遍歷整個 data_loader, 並使用 torch.no_grad 避免計算梯度
    with torch.no_grad():
        for batch in data_loader:
            loss_batch, accu_batch = compute_loss_and_accuracy(batch, net, loss_function, device)
            # .item() 方法可以將單一元素的 Tensor 轉換為 Python 的標準純數值 (float)
            loss_total += loss_batch.item()
            accu_total += accu_batch.item()

    # 計算平均 loss 與準確率
    loss = loss_total / len(data_loader)
    accu = accu_total / len(data_loader)
    return loss, accu


def fit(epochs, lr, net, train_loader, val_loader, writer, opt_func=torch.optim.AdamW, device=None):
    """ 對網路進行訓練和測試, 並藉由 tensorboard writer 進行紀錄 """
    # optimizer 必須將 learning rate 和要訓練的參數傳入, 這邊使用 net.parameters() 來取得所有參數
    optimizer = opt_func(net.parameters(), lr)
    # 定義 loss function
    loss_func = nn.CrossEntropyLoss()
    
    for epoch in range(epochs):
        loss_train = 0
        train_accu_total = 0
        # 將網路用 .train() 方法將其設定為訓練模式
        net.train()
        
        # 使用 tqdm 包裝 train_loader，建立視覺化進度條
        # leave=False 可以在該 epoch 結束後收起進度條，保持版面整潔
        pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{epochs}', leave=False)
        
        for batch in pbar:
            # 使用 compute_loss_and_accuracy 計算該 batch 的 loss 和 accuracy
            loss, accu_train = compute_loss_and_accuracy(batch, net, loss_func, device)
            # 使用 .backward() 進行反向傳播
            loss.backward()
            # 使用 .step() 進行 optimizer 的更新
            optimizer.step()
            # 使用 .zero_grad() 將梯度歸零, 避免累加
            optimizer.zero_grad()
            
            # 紀錄 loss 和 accuracy, .item() 方法將 tensor 從 GPU 中取出轉成 Python 數值
            loss_train += loss.item()
            train_accu_total += accu_train.item()
            
            # 在進度條後方即時顯示目前的 batch loss，讓訓練過程更有感
            pbar.set_postfix({'loss': f'{loss.item():.4f}'})
            
        # 訓練完一個 epoch 後, 計算 validation set 的 loss 和 accuracy
        loss_val, accu_val = evaluate(net, val_loader, loss_func, device)
        
        # 計算當前 Epoch 的平均訓練 Loss 和 Accuracy
        avg_loss_train = loss_train / len(train_loader)
        avg_accu_train = train_accu_total / len(train_loader)

        if epoch % 10 == 0 or epoch == epochs - 1:
            # 印出訓練和測試的平均 loss
            print(f'[{epoch+1}/{epochs}] Training loss: {avg_loss_train:.4f}, Validation loss: {loss_val:.4f}')

        # tensorboard 相關紀錄，包含 training 和 testing 的 loss, accuracy
        writer.add_scalar('loss/train', avg_loss_train, epoch)
        writer.add_scalar('loss/valid', loss_val, epoch)
        writer.add_scalar('accu/train', avg_accu_train, epoch)
        writer.add_scalar('accu/valid', accu_val, epoch)
        
        # 將網路的參數用 histogram 方法紀錄到 tensorboard 中
        for index, t in enumerate(net.parameters()):
            writer.add_histogram(f'v/{index:02d}', t, epoch)

初始化並使用torchinfo印出全連接網路模型

In [ ]:
# 定義device, 若有GPU就使用GPU, 否則才會用cpu
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# 由於torchinfo會要求定義輸入，這邊直接用iter把dataloader轉換成迭代器，再使用next方法取出一個batch
dataiter = iter(trainloader)
images, labels = next(dataiter)

# 由於輸入資料和類神經網路必須要在相同裝置上進行運算, 因此使用 .to(device) 方法將輸入影像轉換到計算裝置上
images = images.to(device)

# 初始化全連接網路, 並將其轉換到計算裝置上
model_fc = FullyConnectedModel().to(device)
# 透過 torchinfo.summary 方法可視化網路架構
summary(model_fc, input_data=images)


初始化卷積網路模型以及使用資料擴增的兩種網路模型

In [ ]:
model_cnn = ConvolutionalModel().to(device)
model_fc_aug = FullyConnectedModel().to(device)
model_cnn_aug = ConvolutionalModel().to(device)

summary(model_cnn, input_data=images)


定義網路訓練超參數

In [ ]:
# 創建 tensorboard 的 log 目錄
model_dir = 'models'
os.makedirs(model_dir, exist_ok=True)

# 定義訓練的超參數
epochs = 50
lr = 1e-3

# 定義要訓練的模型和相關參數，這邊會連續使用 FCN, CNN, Augmented FCN, Augmented CNN 來比較網路架構和是否使用影像增強方法對結果的影響 
model_seq = zip(
    (model_cnn, model_fc, model_cnn_aug, model_fc_aug),
    (False, False, True, True),  # 是否使用 data augmentation
    ('model_cnn', 'model_fc', 'model_cnn_aug', 'model_fc_aug')  # tensorboard紀錄資料夾名稱
)



### 訓練網路模型


進行訓練迴圈

In [ ]:
# 開始訓練模型, model 代表要訓練的模型, aug 代表是否使用 image augmentation, name 代表 log 的目錄名稱
for model, aug, name in model_seq:
    print(f'開始訓練模型: {name}')
    # 創建 tensorboard writer 來記錄訓練過程
    writer = SummaryWriter(os.path.join(model_dir, name))
    # 依照 aug 選擇要使用的 dataloader
    loader_train = trainloader if not aug else trainloader_aug
    # 開始訓練
    fit(epochs, lr, model, loader_train, validloader, writer, device=device)
    # 在 tensorboard 中加入模型的結構，需先將模型轉換成 evaluation 模式
    model.eval()
    writer.add_graph(model, torch.rand_like(images, device=device))
    # 關閉 tensorboard writer
    writer.close()


開啟TensorBoard查看訓練紀錄

In [ ]:
# 載入 TensorBoard 的 notebook 擴充套件
%load_ext tensorboard

# 在 Colab 中啟動並顯示 TensorBoard 儀表板
%tensorboard --logdir models